## **Approach to LIDA Composting Project (Iteration 1)**

#### *Data preprocessing*
- Using the lida_tabletrimmed csv file, we'll gather all the feature sets and filter out the rows to be recorded **daily**. 
- All values will be cleaned and converted to positive values. 

#### *Data Oversampling*
- With the minuscule amount of data we have for training, an oversample technique must be implemented as compensation.
- We will implemented an oversampling technique that generates multiple synthetic data that preserves the time series qualities of composting

#### *Change Point Detection*
#### *Future Forecaster*


---

### **Data Init**

In [1]:
import pandas as pd
import numpy as np

file_path = "data/lida_tabletrimmed.csv"
df = pd.read_csv(file_path)

### **Filtering rows by day**

In [2]:
# filtering out the rows by date. (daily)

df['time_stamp'] = pd.to_datetime(df['time_stamp'])
df_daily = df.groupby(df['time_stamp'].dt.date).first().reset_index(drop=True)

/tmp/ipykernel_18209/1057245352.py:3: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['time_stamp'] = pd.to_datetime(df['time_stamp'])


In [3]:
df_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   moisture_active1     21 non-null     int64         
 1   moisture_active2     21 non-null     int64         
 2   oxygen               21 non-null     float64       
 3   lid                  21 non-null     object        
 4   co2                  21 non-null     int64         
 5   time_stamp           21 non-null     datetime64[ns]
 6   device_id            21 non-null     object        
 7   temperature_curing1  21 non-null     int64         
 8   temperature_curing2  21 non-null     int64         
 9   moisture_curing1     21 non-null     int64         
 10  moisture_curing2     21 non-null     int64         
 11  automation_active    21 non-null     bool          
 12  methane              21 non-null     int64         
 13  temperature_active1  21 non-null     

### **Dropping columns we don't need**

In [4]:
# Dropping unnecessary cols
df_daily.drop(columns=['device_id', 'lid', 'automation_active'], inplace=True)

In [5]:
numeric_cols = df_daily.select_dtypes(include=[np.number]).columns
df_daily[numeric_cols] = df_daily[numeric_cols].abs()

In [6]:
df_daily.head(21)

,moisture_active1,moisture_active2,oxygen,co2,time_stamp,temperature_curing1,temperature_curing2,moisture_curing1,moisture_curing2,methane,temperature_active1,temperature_active2,temperature_active3,temperature_active4
0,14,15,10.429501,6473,2024-06-28 18:03:00,0,0,134,14,965,15,15,25,14
1,15,10,14.447586,6637,2024-06-29 00:00:00,0,0,104,22,1190,16,16,16,16
2,18,16,6.715520,6534,2024-06-30 14:40:00,0,0,145,27,1387,18,19,21,0
3,18,20,11.607240,6610,2024-07-01 00:00:00,0,0,28,26,789,18,19,21,18
4,29,26,11.753019,6422,2024-07-02 00:00:00,0,0,96,28,777,20,22,26,19
5,26,23,11.071794,6483,2024-07-03 00:00:00,0,0,102,28,886,24,25,30,23
6,30,20,10.902768,6368,2024-07-04 00:00:00,0,0,149,42,941,26,25,29,26
7,20,22,11.018209,6156,2024-07-05 07:21:00,0,0,52,56,788,25,25,25,25
8,22,25,10.552896,6190,2024-07-06 00:00:00,0,0,67,58,926,22,21,25,23
9,54,9,23.418449,6463,2024-07-07 16:06:00,0,0,125,40,1051,16,25,25,16


### **Oversampling implementation**
1. Interpolate the Base Data
2. Generate Multiple Synthetic Runs
3. Add Controlled Random Variation
4. Keep Time Continuous across runs
5. Label and combine

In [ ]:
import pandas as pd
import numpy as np

def generate_synthetic_compost(df, time_col='time_stamp', freq='D', noise_factor=0.02, n_runs=10):
    """
    Generate synthetic composting runs from a small dataset,
    continuous in time and without NaNs, preserving compost cycle shape.
    """
    
    # Columns that should not be noised
    non_numeric_cols = [time_col, 'device_id', 'automation_active', 'lid']
    
    # Ensure datetime
    df[time_col] = pd.to_datetime(df[time_col])
    
    # Sort by time and set index for resampling
    df = df.sort_values(time_col).set_index(time_col)

    # Interpolate to higher resolution
    df_interp = df.resample(freq).interpolate(method='linear')
    
    synthetic_runs = []
    current_start = df_interp.index.min()
    
    for run_id in range(n_runs):
        df_aug = df_interp.copy()
        
        # Add noise to numeric columns only
        for col in df_aug.columns:
            if col not in non_numeric_cols and pd.api.types.is_numeric_dtype(df_aug[col]):
                noise = np.random.normal(
                    0, noise_factor * df_aug[col].std(), size=len(df_aug)
                )
                df_aug[col] += noise
        
        # Assign continuous timestamps for this run
        run_start_date = current_start
        df_aug.index = pd.date_range(
            start=run_start_date, 
            periods=len(df_aug), 
            freq=freq
        )
        
        # Drop NaNs (should be rare after interpolation)
        df_aug = df_aug.dropna()
        
        # Add run_id and reset index to restore time_col
        df_aug['run_id'] = run_id
        df_aug = df_aug.reset_index().rename(columns={'index': time_col})
        
        synthetic_runs.append(df_aug)
        
        # Move start date for next run
        current_start = df_aug[time_col].max() + pd.Timedelta(days=1)
    
    # Combine all runs
    return pd.concat(synthetic_runs, ignore_index=True)


In [8]:
synthetic_df = generate_synthetic_compost(df_daily, time_col='time_stamp', freq='1D', noise_factor=0.03, n_runs=5)

In [9]:
synthetic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   time_stamp           265 non-null    datetime64[ns]
 1   moisture_active1     265 non-null    float64       
 2   moisture_active2     265 non-null    float64       
 3   oxygen               265 non-null    float64       
 4   co2                  265 non-null    float64       
 5   temperature_curing1  265 non-null    float64       
 6   temperature_curing2  265 non-null    float64       
 7   moisture_curing1     265 non-null    float64       
 8   moisture_curing2     265 non-null    float64       
 9   methane              265 non-null    float64       
 10  temperature_active1  265 non-null    float64       
 11  temperature_active2  265 non-null    float64       
 12  temperature_active3  265 non-null    float64       
 13  temperature_active4  265 non-null  

In [10]:
# synthetic_df.to_csv("oversampled.csv")